<div style="text-align: center;">
    <h1>Support Vector Machine — Clasificación de Enfermedad Cardiovascular</h1>
</div>

## 1. Importar librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, kendalltau, loguniform
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,fbeta_score,roc_auc_score,confusion_matrix,make_scorer,precision_recall_curve,RocCurveDisplay

## 2. Cargar datos

Se carga directamente la base de datos ya limpia y preprocesada desde el repositorio GitHub del proyecto.

In [3]:
try:
    data = pd.read_csv("https://raw.githubusercontent.com/SantCorrea802/Cardiovascular_Disease_Proyecto_Modelos_II/main/dataset/data_cleaned.csv")
except FileNotFoundError:
    print("No se ha encontrado el archivo de datos. Verifique el directorio.")
    exit(1)

In [4]:
data.head()

,age,gender,height,weight,systolic_blood_pressure,diastolic_blood_pressure,cholesterol,glucose,smoke,alcohol_intake,physical_activity,bmi,cardiovascular_disease
0,50,2,168,62.0,110,80,1,1,0,0,1,1,0
1,55,1,156,85.0,140,90,3,1,0,0,1,3,1
2,52,1,165,64.0,130,70,3,1,0,0,0,1,1
3,48,2,169,82.0,150,100,1,1,0,0,1,2,1
4,48,1,156,56.0,100,60,1,1,0,0,0,1,0


## 3. Separación Train / Validation / Test

Se usa la misma estrategia del EDA: **80% train — 10% val — 10% test**, con estratificación para mantener la proporción de clases.

In [5]:
features = data.drop(columns=["cardiovascular_disease"])
target = data["cardiovascular_disease"]

#separar train del resto
X_train, X_temp, y_train, y_temp = train_test_split(features, target, test_size=0.20, random_state=42, stratify=target)
#separar test del val
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

# juntar data_train, data_test y data_val para el EDA
data_train = pd.concat([X_train, y_train], axis=1)
data_val = pd.concat([X_val, y_val], axis=1)
data_test = pd.concat([X_test, y_test], axis=1)

(54871, 12) (6859, 12) (6859, 12)
(54871,) (6859,) (6859,)


Separaremos caracteristicas según su categoria o clasificiación

In [6]:
categorical = ["gender", "cholesterol", "glucose", "smoke", "alcohol_intake", "physical_activity","bmi"]
nominal = ["gender", "smoke", "alcohol_intake", "physical_activity"]
ordinal = ["cholesterol", "glucose", "bmi"]
binary = ["gender","smoke", "alcohol_intake", "physical_activity"]
numerical = ["age", "height", "weight", "systolic_blood_pressure", "diastolic_blood_pressure"]

In [7]:
data["bmi"].unique()

array([1, 3, 2, 4, 0, 5])

## 4. Preprocesamiento obligatorio para SVM

Las maquinas de soporte vectorial dependen de distancias, márgenes y productos internos. Por eso el escalado es obligatorio

In [8]:
standar = numerical
onehot = nominal
# ordinal es la misma ordinal

# categorias para la ordinal
categorias = [
    [1,2,3],
    [1,2,3],
    [0,1,2,3,4,5]
]

ordinal_pipeline = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(
            categories=categorias,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-2
        )),
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), standar),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=True), onehot),
        ("ord", ordinal_pipeline, ordinal)
    ],
    remainder="drop"
)

## 5. Modelos Base
Antes de buscar hiperparámetros, entrenar modelos base

Usar **DummyClassifier** para responder a la pregunta ¿Mi modelo real es mejor que una estrategia tonta?

In [9]:
dummy_model = Pipeline(
    steps=[
        ("preprocess", preprocessor, ),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]
)

dummy_model_trained = dummy_model.fit(X_train, y_train)

Usar **SVM Lineal**

In [10]:
linear_svm_base = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LinearSVC(
            C=1.0,
            class_weight=None,
            max_iter=20_000,
            random_state=42
        ))
    ]
)

linear_svm_trained = linear_svm_base.fit(X_train, y_train)

Usar **SVM RBF**

In [11]:
rbf_svm_base = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            class_weight=None,
            cache_size=1000,
            random_state=42
        ))
    ]
)

svm_rbf_trained = rbf_svm_base.fit(X_train, y_train)

## 6. Función de evaluación

In [12]:
def get_scores(model, X, y, model_name):
    y_pred = model.predict(X)

    if hasattr(model, "decision_function"):
        y_score = model.decision_function(X)
    elif hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X)[:, 1]
    else:
        y_score = None

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    scores = {
        "model": model_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "f2": fbeta_score(y, y_pred, beta=2, zero_division=0),
        "roc_auc": roc_auc_score(y, y_score) if y_score is not None else None,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    return scores, cm

In [13]:
def confusion_matrix_table(model, X, y):
    y_pred = model.predict(X)

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100

    cm_combined = pd.DataFrame(
        [
            [
                f"{cm[i, j]} ({cm_percent[i, j]:.2f}%)"
                for j in range(cm.shape[1])
            ]
            for i in range(cm.shape[0])
        ],
        index=["Real 0: Sin enfermedad", "Real 1: Con enfermedad"],
        columns=["Predicho 0: Sin enfermedad", "Predicho 1: Con enfermedad"]
    )

    return cm_combined

In [14]:
models = {
    "Dummy": dummy_model_trained,
    "Linear SVM base": linear_svm_trained,
    "RBF SVM base": svm_rbf_trained
}

results = []
confusion_matrices = {}

for model_name, model in models.items():
    train_scores, train_cm = get_scores(
        model,
        X_train,
        y_train,
        model_name + " - train"
    )

    val_scores, val_cm = get_scores(
        model,
        X_val,
        y_val,
        model_name + " - validation"
    )

    results.append(train_scores)
    results.append(val_scores)

    confusion_matrices[model_name + " - train"] = train_cm
    confusion_matrices[model_name + " - validation"] = val_cm

base_results = pd.DataFrame(results)

base_results = base_results.sort_values(
    by=["f2", "FN_%", "recall"],
    ascending=[False, True, False]
).round(4)

base_results

,model,accuracy,precision,recall,f1,f2,roc_auc,TN,FP,FN,TP
0,Dummy - train,0.504966,0.000000,0.000000,0.000000,0.000000,0.500000,27708,0,27163,0
1,Dummy - validation,0.504884,0.000000,0.000000,0.000000,0.000000,0.500000,3463,0,3396,0
2,Linear SVM base - train,0.725265,0.755344,0.658212,0.703441,0.675587,0.789337,21917,5791,9284,17879
3,Linear SVM base - validation,0.731010,0.760497,0.666667,0.710497,0.683534,0.796289,2750,713,1132,2264
4,RBF SVM base - train,0.736418,0.766471,0.672422,0.716373,0.689339,0.798232,22143,5565,8898,18265
5,RBF SVM base - validation,0.739758,0.768590,0.678740,0.720876,0.694989,0.797395,2769,694,1091,2305


In [15]:
confusion_matrix_table(svm_rbf_trained, X_val, y_val)

,Predicho 0: Sin enfermedad,Predicho 1: Con enfermedad
Real 0: Sin enfermedad,2769 (79.96%),694 (20.04%)
Real 1: Con enfermedad,1091 (32.13%),2305 (67.87%)


In [16]:
confusion_matrix_table(linear_svm_trained, X_val, y_val)

,Predicho 0: Sin enfermedad,Predicho 1: Con enfermedad
Real 0: Sin enfermedad,2750 (79.41%),713 (20.59%)
Real 1: Con enfermedad,1132 (33.33%),2264 (66.67%)


In [17]:
confusion_matrix_table(dummy_model_trained, X_val, y_val)

,Predicho 0: Sin enfermedad,Predicho 1: Con enfermedad
Real 0: Sin enfermedad,3463 (100.00%),0 (0.00%)
Real 1: Con enfermedad,3396 (100.00%),0 (0.00%)


## 7. Busqueda de hiperparametros

In [18]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

### Busqueda para linear SVM

In [19]:
f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    zero_division=0
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "f2": f2_scorer
}

In [20]:
linear_svm = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LinearSVC(
            max_iter=30000,
            random_state=42
        ))
    ]
)

linear_param_grid = {
    "model__C": np.logspace(-3, 3, 7),
    "model__class_weight": [None, "balanced"]
}


linear_search = GridSearchCV(
    estimator=linear_svm,
    param_grid=linear_param_grid,
    scoring=scoring,
    refit="f2",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

linear_search.fit(X_train, y_train)

linear_search.best_params_, linear_search.best_score_

({'model__C': np.float64(0.001), 'model__class_weight': 'balanced'},
 np.float64(0.6806438095155525))

## Tabla resultados de Linera SVM

In [21]:
linear_results = pd.DataFrame(linear_search.cv_results_)

linear_summary = linear_results[
    [
        "param_model__C",
        "param_model__class_weight",
        "mean_test_accuracy",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_f1",
        "mean_test_f2",
        "mean_test_roc_auc",
        "std_test_f2",
        "rank_test_f2"
    ]
].sort_values("rank_test_f2")

linear_summary.head(10)

,param_model__C,param_model__class_weight,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_f2,mean_test_roc_auc,std_test_f2,rank_test_f2
1,0.001,balanced,0.725556,0.751998,0.664875,0.705754,0.680644,0.789045,0.004102,1
5,0.100,balanced,0.725411,0.752082,0.664286,0.705460,0.680164,0.789083,0.004082,2
7,1.000,balanced,0.725411,0.752082,0.664286,0.705460,0.680164,0.789084,0.004082,2
9,10.000,balanced,0.725411,0.752082,0.664286,0.705460,0.680164,0.789084,0.004082,2
11,100.000,balanced,0.725411,0.752082,0.664286,0.705460,0.680164,0.789084,0.004082,2
13,1000.000,balanced,0.725411,0.752082,0.664286,0.705460,0.680164,0.789084,0.004082,2
3,0.010,balanced,0.725392,0.752050,0.664286,0.705446,0.680159,0.789080,0.004178,7
6,1.000,None,0.724773,0.754766,0.657733,0.702910,0.675087,0.789094,0.004495,8
10,100.000,None,0.724773,0.754766,0.657733,0.702910,0.675087,0.789094,0.004495,8
12,1000.000,None,0.724773,0.754766,0.657733,0.702910,0.675087,0.789094,0.004495,8


### Busqueda para RBF SVM

In [22]:
rbf_svm = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", SVC(
            kernel="rbf",
            cache_size=500,
            random_state=42
        ))
    ]
)

rbf_param_distributions = {
    "model__C": loguniform(1e-2, 1e3),
    "model__gamma": loguniform(1e-4, 1e1),
    "model__class_weight": [None, "balanced"]
}

rbf_search = RandomizedSearchCV(
    estimator=rbf_svm,
    param_distributions=rbf_param_distributions,
    n_iter=16,
    scoring=scoring,
    refit="f2",
    cv=cv,
    n_jobs=2,
    random_state=42,
    return_train_score=False,
    verbose=2
)

rbf_search.fit(X_train, y_train)

rbf_search.best_params_, rbf_search.best_score_

Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END model__C=0.7459343285726545, model__class_weight=None, model__gamma=0.0008263688714158015; total time= 2.6min
[CV] END model__C=79.15074397656205, model__class_weight=None, model__gamma=0.0006026889128682511; total time= 2.6min


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END model__C=0.7459343285726545, model__class_weight=None, model__gamma=0.0008263688714158015; total time= 2.7min
[CV] END model__C=79.15074397656205, model__class_weight=None, model__gamma=0.0006026889128682511; total time= 2.7min
[CV] END model__C=0.7459343285726545, model__class_weight=None, model__gamma=0.0008263688714158015; total time= 2.6min
[CV] END model__C=0.7459343285726545, model__class_weight=None, model__gamma=0.0008263688714158015; total time= 2.6min
[CV] END model__C=79.15074397656205, model__class_weight=None, model__gamma=0.0006026889128682511; total time= 2.7min
[CV] END model__C=0.060252157362038566, model__class_weight=None, model__gamma=0.019780827689353773; total time= 2.8min
[CV] END model__C=0.4661686413912768, model__class_weight=balanced, model__gamma=0.3470266988650412; total time= 3.4min
[CV] END model__C=0.7459343285726545, model__class_weight=None, model__gamma=0.0008263688714158015; total time= 2.7min
[CV] END model__C=79.15074397656205, model__clas

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

({'model__C': np.float64(0.012674255898937233),
  'model__class_weight': 'balanced',
  'model__gamma': np.float64(0.40737451960583815)},
 np.float64(0.7331046933420478))

### Tabla de resultados RBF

In [23]:
rbf_results = pd.DataFrame(rbf_search.cv_results_)

rbf_summary = rbf_results[
    [
        "param_model__C",
        "param_model__gamma",
        "param_model__class_weight",
        "mean_test_accuracy",
        "mean_test_precision",
        "mean_test_recall",
        "mean_test_f1",
        "mean_test_f2",
        "mean_test_roc_auc",
        "std_test_f2",
        "rank_test_f2"
    ]
].sort_values("rank_test_f2")

rbf_summary.head(10)

,param_model__C,param_model__gamma,param_model__class_weight,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_f2,mean_test_roc_auc,std_test_f2,rank_test_f2
4,0.012674,0.407375,balanced,0.708207,0.690300,0.744653,0.716442,0.733105,0.773454,0.004168,1
3,0.466169,0.347027,balanced,0.730823,0.750435,0.683577,0.715441,0.695975,0.780279,0.004972,2
5,492.905365,0.000811,balanced,0.729383,0.749582,0.680779,0.713524,0.693509,0.791820,0.003358,3
6,0.082608,0.114358,balanced,0.730422,0.755495,0.673342,0.712052,0.688309,0.789880,0.005670,4
12,10.907476,0.017885,None,0.731078,0.759865,0.667783,0.710850,0.684367,0.791628,0.005118,5
10,84.310139,0.008172,None,0.731279,0.760651,0.667047,0.710777,0.683876,0.791540,0.005369,6
9,739.226614,0.019070,None,0.730112,0.760518,0.663844,0.708895,0.681158,0.786961,0.005239,7
15,0.142715,0.015877,balanced,0.729001,0.759232,0.662740,0.707709,0.680023,0.791598,0.004607,8
1,79.150744,0.000603,None,0.728819,0.761049,0.659169,0.706454,0.677302,0.791112,0.003456,9
11,824.431219,0.000171,None,0.728800,0.762874,0.656113,0.705476,0.675005,0.791065,0.003631,10


### Busqueda refinada

In [24]:
best_C = rbf_search.best_params_["model__C"]
best_gamma = rbf_search.best_params_["model__gamma"]
best_class_weight = rbf_search.best_params_["model__class_weight"]

refined_rbf_param_distributions = {
    "model__C": loguniform(
        max(best_C / 10, 1e-4),
        best_C * 10
    ),
    "model__gamma": loguniform(
        max(best_gamma / 10, 1e-6),
        best_gamma * 10
    ),
    "model__class_weight": [best_class_weight, None, "balanced"]
}

In [ ]:
rbf_refined_search = RandomizedSearchCV(
    estimator=rbf_svm,
    param_distributions=refined_rbf_param_distributions,
    n_iter=12,
    scoring=scoring,
    refit="f2",
    cv=cv,
    n_jobs=-1,
    random_state=123,
    return_train_score=False,
    verbose=2
)

rbf_refined_search.fit(X_train, y_train)

rbf_refined_search.best_params_, rbf_refined_search.best_score_

Fitting 5 folds for each of 12 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Crear tabla de modelos base

In [ ]:
base_models = {
    "Dummy base": dummy_model_trained,
    "Linear SVM base": linear_svm_trained,
    "RBF SVM base": svm_rbf_trained
}

base_results = []

for model_name, model in base_models.items():
    scores, cm = get_scores(
        model=model,
        X=X_val,
        y=y_val,
        model_name=model_name
    )

    base_results.append(scores)

base_validation_results = pd.DataFrame(base_results).sort_values(
    by=["f2", "FN", "recall"],
    ascending=[False, True, False]
)

base_validation_results

### Comparar en validación

In [ ]:
best_linear_svm = linear_search.best_estimator_
best_rbf_svm = rbf_search.best_estimator_
best_rbf_refined_svm = rbf_refined_search.best_estimator_

models_tuned = {
    "Linear SVM tuned": best_linear_svm,
    "RBF SVM tuned": best_rbf_svm,
    "RBF SVM refined": best_rbf_refined_svm
}

tuned_results = []

for model_name, model in models_tuned.items():
    scores, cm = get_scores(
        model,
        X_val,
        y_val,
        model_name
    )

    tuned_results.append(scores)

tuned_results = pd.DataFrame(tuned_results).sort_values(
    by=["f2", "FN", "recall"],
    ascending=[False, True, False]
)

tuned_results

### Unir

La siguiente celda une:

Modelos base:
- Dummy base
- Linear SVM base
- RBF SVM base

Modelos optimizados:
- Linear SVM tuned
- RBF SVM tuned sample

In [ ]:
all_validation_results = pd.concat(
    [
        base_validation_results,
        tuned_results
    ],
    ignore_index=True
).sort_values(
    by=["f2", "FN", "recall"],
    ascending=[False, True, False]
)

all_validation_results

## Curvas ROC en validación

Se grafican las curvas ROC de los modelos SVM evaluados en el conjunto de validación.  
La curva ROC permite analizar la capacidad del modelo para separar pacientes con y sin enfermedad cardiovascular considerando distintos umbrales de decisión

El eje vertical representa la tasa de verdaderos positivos, también llamada recall o sensibilidad:

$$TPR = \frac{TP}{TP + FN}$$

El eje horizontal representa la tasa de falsos positivos:

$$FPR = \frac{FP}{FP + TN}$$

Un mejor modelo tendrá una curva más cercana a la esquina superior izquierda y un mayor valor de AUC-ROC.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_estimator(
    linear_svm_trained,
    X_val,
    y_val,
    ax=ax,
    name="Linear SVM base"
)

RocCurveDisplay.from_estimator(
    svm_rbf_trained,
    X_val,
    y_val,
    ax=ax,
    name="RBF SVM base"
)

RocCurveDisplay.from_estimator(
    best_linear_svm,
    X_val,
    y_val,
    ax=ax,
    name="Linear SVM tuned"
)

RocCurveDisplay.from_estimator(
    best_rbf_svm,
    X_val,
    y_val,
    ax=ax,
    name="RBF SVM tuned"
)

RocCurveDisplay.from_estimator(
    best_rbf_refined_svm,
    X_val,
    y_val,
    ax=ax,
    name="RBF SVM refined"
)

plt.title("Curvas ROC de modelos SVM en validación")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos / Recall")
plt.grid(True)
plt.show()

### Ajuste de umbral de decisión en validación

candidato

In [ ]:
#candidate_model = best_rbf_svm
#candidate_model_name = "RBF SVM tuned"
#candidate_model = best_linear_svm
#candidate_model_name = "Linear SVM tuned"
#candidate_model = best_rbf_refined_svm
#candidate_model_name = "RBF SVM refined"

In [ ]:
scores_val = candidate_model.decision_function(X_val)

# Umbrales candidatos. Incluye el threshold default de SVM: 0.0
thresholds = np.unique(
    np.r_[
        np.quantile(scores_val, np.linspace(0.01, 0.99, 99)),
        0.0
    ]
)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (scores_val >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        y_pred_threshold,
        labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "model": candidate_model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, y_pred_threshold),
        "precision": precision_score(y_val, y_pred_threshold, zero_division=0),
        "recall": recall_score(y_val, y_pred_threshold, zero_division=0),
        "f1": f1_score(y_val, y_pred_threshold, zero_division=0),
        "f2": fbeta_score(y_val, y_pred_threshold, beta=2, zero_division=0),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

threshold_results = pd.DataFrame(threshold_results)

threshold_results_sorted = threshold_results.sort_values(
    by=["f2", "FN", "recall"],
    ascending=[False, True, False]
)

threshold_results_sorted.head(10)

In [ ]:
### Guardar mejor treshold

In [ ]:
best_threshold = threshold_results_sorted.iloc[0]["threshold"]

best_threshold

### Comparar default (0.0) vs treshold ajustado

permite decir si con el threshold ajustado gané recall / reduje FN, pero aumenté o no aumenté FP

In [ ]:
default_result = threshold_results.loc[
    threshold_results["threshold"].abs().idxmin()
]

adjusted_result = threshold_results_sorted.iloc[0]

pd.DataFrame([default_result, adjusted_result], index=["Default threshold", "Adjusted threshold"])

### Matriz de confusión con treshold ajustado

In [ ]:
y_val_pred_adjusted = (scores_val >= best_threshold).astype(int)

cm_val_adjusted = confusion_matrix(
    y_val,
    y_val_pred_adjusted,
    labels=[0, 1]
)

cm_val_adjusted_df = pd.DataFrame(
    cm_val_adjusted,
    index=["Real 0: Sin enfermedad", "Real 1: Con enfermedad"],
    columns=["Predicho 0: Sin enfermedad", "Predicho 1: Con enfermedad"]
)

cm_val_adjusted_df

In [ ]:
cm_percent = cm_val_adjusted / cm_val_adjusted.sum(axis=1, keepdims=True) * 100

cm_combined = pd.DataFrame(
    [
        [
            f"{cm_val_adjusted[i, j]} ({cm_percent[i, j]:.2f}%)"
            for j in range(cm_val_adjusted.shape[1])
        ]
        for i in range(cm_val_adjusted.shape[0])
    ],
    index=["Real 0: Sin enfermedad", "Real 1: Con enfermedad"],
    columns=["Predicho 0: Sin enfermedad", "Predicho 1: Con enfermedad"]
)

cm_combined